# Φάση Δ: Advanced Technique

Εμείς επιλέξαμε το K-Means


In [2]:
# =====================================================================
# Φάση Δ: Advanced Technique (Association Rules & Clustering)
# =====================================================================

from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.ml.fpm import FPGrowth
from pyspark.ml.clustering import KMeans
from pyspark.ml.feature import PCA

print("Αρχικοποίηση SparkSession...")
spark = SparkSession.builder \
    .appName("Stroke_Advanced_Techniques") \
    .master("local[*]") \
    .getOrCreate()

# =====================================================================
# 1. ASSOCIATION RULES (FP-Growth) - Ζητούμενο Εκφώνησης
# =====================================================================
print("\n--- 1. ASSOCIATION RULES (FP-Growth) ---")
print("Φόρτωση Silver Layer για εξαγωγή κατηγορικών χαρακτηριστικών...")
df_silver = spark.read.parquet("../data/train_silver.parquet")

# Μετατροπή των στηλών σε Array (Καλάθι χαρακτηριστικών)
transactions_df = df_silver.withColumn(
    "items", 
    F.array_remove(F.array(
        F.when(F.col("stroke") == 1, "Has Stroke").otherwise(""),
        F.when(F.col("hypertension") == 1, "Has Hypertension").otherwise(""),
        F.when(F.col("heart_disease") == 1, "Has Heart Disease").otherwise(""),
        F.when(F.col("smoking_status") == "smokes", "Smoker").otherwise(""),
        F.when(F.col("age") > 65, "Senior (>65)").otherwise("")
    ), "")
)
transactions_df = transactions_df.filter(F.size("items") > 0).select("items")

print("Εκπαίδευση FP-Growth...")
fpGrowth = FPGrowth(itemsCol="items", minSupport=0.05, minConfidence=0.2)
fpm_model = fpGrowth.fit(transactions_df)

print("\nΚανόνες Συσχέτισης (Που οδηγούν σε Εγκεφαλικό):")
rules = fpm_model.associationRules
rules.filter(F.array_contains(F.col("consequent"), "Has Stroke")).sort(F.col("confidence").desc()).show(truncate=False)

# =====================================================================
# 2. CLUSTERING (K-MEANS) & PCA - Αρχική υλοποίηση φοιτητή
# =====================================================================
print("\n--- 2. CLUSTERING (K-Means) ---")
train_gold = spark.read.parquet("../data/train_gold.parquet")

kmeans = KMeans(featuresCol="features", predictionCol="cluster", k=3, seed=42)
kmeans_model = kmeans.fit(train_gold)
kmeans_preds = kmeans_model.transform(train_gold)

print("Εφαρμογή PCA (2D) για οπτικοποίηση Clusters...")
pca = PCA(k=2, inputCol="features", outputCol="pca_features")
pca_model = pca.fit(kmeans_preds)
kmeans_pca = pca_model.transform(kmeans_preds)

print("Αποθήκευση K-Means προβλέψεων...")
kmeans_pca.select("stroke", "cluster", "pca_features") \
    .write.mode("overwrite").parquet("../data/preds_kmeans.parquet")

print("Η Φάση Δ ολοκληρώθηκε!")

Αρχικοποίηση SparkSession...

--- 1. ASSOCIATION RULES (FP-Growth) ---
Φόρτωση Silver Layer για εξαγωγή κατηγορικών χαρακτηριστικών...
Εκπαίδευση FP-Growth...

Κανόνες Συσχέτισης (Που οδηγούν σε Εγκεφαλικό):
+----------+----------+----------+----+-------+
|antecedent|consequent|confidence|lift|support|
+----------+----------+----------+----+-------+
+----------+----------+----------+----+-------+


--- 2. CLUSTERING (K-Means) ---
Εφαρμογή PCA (2D) για οπτικοποίηση Clusters...


Αποθήκευση K-Means προβλέψεων...
Η Φάση Δ ολοκληρώθηκε!


26/06/03 18:24:48 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/06/03 18:24:48 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/06/03 18:24:48 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/06/03 18:24:48 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/06/03 18:24:48 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/06/03 18:24:48 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/06/03 18:24:48 WARN MemoryManager: Total allocation exceeds 95.